# Traditional ML Strategy Contract Replay

This notebook validates the broader contract-first approach for non-`optimal_trader` ML strategies.

The notebook no longer owns option backtest mechanics. It converts a model `scored_panel` into the standard artifacts:

`scored_panel -> action_tape -> trade_list -> strategy_artifacts_manifest.json`

Options, WFO, Monte Carlo, and cross-framework reports should consume `trade_list` or the manifest output.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "quant_orchestrator").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.platforms.backtesting_frameworks.scored_panel_replay import (
    ScoredPanelTopKReplayConfig,
    replay_scored_panel_top_k,
)
from quant_orchestrator.artifact_contracts import read_strategy_artifacts

REPO_ROOT


## Input

Set `SCORED_PANEL_PATH` to a parquet panel with `date`, `symbol`, `close`, and the configured score column. If you are running from an upstream notebook, define `scored_panel` in memory before executing this notebook.

In [ ]:
SCORED_PANEL_PATH = None
OUTPUT_DIR = REPO_ROOT / "artifacts" / "traditional_ml_contract_replay"

SCORE_COL = "prob_buy"
TOP_K = 20
THRESHOLD = 0.50
INITIAL_BALANCE = 100_000.0
FEE_BPS = 5.0
SLIPPAGE_BPS = 5.0

if SCORED_PANEL_PATH:
    scored_panel = pd.read_parquet(SCORED_PANEL_PATH)
elif "scored_panel" not in globals():
    raise RuntimeError("Provide SCORED_PANEL_PATH or define scored_panel before running this notebook.")

display(scored_panel.head())


In [ ]:
result = replay_scored_panel_top_k(
    scored_panel,
    config=ScoredPanelTopKReplayConfig(
        score_col=SCORE_COL,
        threshold=THRESHOLD,
        top_k=TOP_K,
        initial_balance=INITIAL_BALANCE,
        fee_bps=FEE_BPS,
        slippage_bps=SLIPPAGE_BPS,
        strategy_name="traditional_ml.scored_panel_top_k",
        output_dir=OUTPUT_DIR,
        metadata={"notebook": "traditional_ml_synthetic_options_backtest"},
    ),
)
bundle = read_strategy_artifacts(OUTPUT_DIR)

assert bundle.scored_panel is not None and not bundle.scored_panel.empty
assert bundle.action_tape is not None
assert bundle.trade_list is not None

result.summary


In [ ]:
display(pd.DataFrame([result.summary.get("performance", {})]))
display(result.rule_replay.action_tape.head(25))
display(result.rule_replay.trade_list.head(25))


## Downstream Consumers

Use `OUTPUT_DIR / "strategy_artifacts_manifest.json"` or `OUTPUT_DIR / "trade_list.parquet"` as the input to options replay, WFO, Monte Carlo, or cross-framework comparison. This keeps option mechanics out of the model notebook.

In [ ]:
{
    "manifest": str(OUTPUT_DIR / "strategy_artifacts_manifest.json"),
    "strategy_name": bundle.strategy_name,
    "scored_rows": len(bundle.scored_panel),
    "action_rows": len(bundle.action_tape),
    "trade_list": len(bundle.trade_list),
}
